In [1]:
# ==========================================================
# Notebook 05: Cross-Encoder Re-ranking
# ==========================================================

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
corpus_df = pd.read_csv("data/search_corpus.csv")

corpus_df.head()

,doc_id,title,category,text,char_length,word_count,source,language,version
0,1,Artificial Intelligence for Beginners,AI,Artificial Intelligence is the simulation of h...,76,10,internal_knowledge_base,en,v1
1,2,Machine Learning Basics,ML,Machine Learning is a subset of Artificial Int...,78,12,internal_knowledge_base,en,v1
2,3,Deep Learning Introduction,DL,Deep Learning uses neural networks with multip...,63,9,internal_knowledge_base,en,v1
3,4,Natural Language Processing,NLP,Natural Language Processing helps computers un...,70,8,internal_knowledge_base,en,v1
4,5,Python Programming,Programming,Python is one of the most popular programming ...,63,11,internal_knowledge_base,en,v1


In [3]:
documents = corpus_df["text"].tolist()

titles = corpus_df["title"].tolist()

In [4]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Bi-Encoder Loaded!")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Bi-Encoder Loaded!


In [5]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder Loaded!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vinna\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not insta

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Loaded!


In [6]:
document_embeddings = bi_encoder.encode(documents)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity


def retrieve_candidates(query, documents, embeddings, model, top_k=5):

    query_embedding = model.encode([query])

    scores = cosine_similarity(query_embedding, embeddings)[0]

    results = pd.DataFrame({"document": documents, "bi_score": scores})

    results = results.sort_values(by="bi_score", ascending=False)

    return results.head(top_k)

In [8]:
query = "How do I start learning Artificial Intelligence?"

candidate_docs = retrieve_candidates(
    query=query,
    documents=documents,
    embeddings=document_embeddings,
    model=bi_encoder,
    top_k=5,
)

candidate_docs

,document,bi_score
0,Artificial Intelligence is the simulation of h...,0.521461
4,Python is one of the most popular programming ...,0.448475
1,Machine Learning is a subset of Artificial Int...,0.405643
3,Natural Language Processing helps computers un...,0.292249
2,Deep Learning uses neural networks with multip...,0.233455


In [9]:
query_document_pairs = [(query, doc) for doc in candidate_docs["document"]]

query_document_pairs

[('How do I start learning Artificial Intelligence?',
  'Artificial Intelligence is the simulation of human intelligence by machines.'),
 ('How do I start learning Artificial Intelligence?',
  'Python is one of the most popular programming languages for AI.'),
 ('How do I start learning Artificial Intelligence?',
  'Machine Learning is a subset of Artificial Intelligence that learns from data.'),
 ('How do I start learning Artificial Intelligence?',
  'Natural Language Processing helps computers understand human language.'),
 ('How do I start learning Artificial Intelligence?',
  'Deep Learning uses neural networks with multiple hidden layers.')]

In [10]:
cross_scores = cross_encoder.predict(query_document_pairs)

cross_scores

array([ -0.20730585, -10.088572  ,  -0.01495757, -10.786014  ,
        -9.772631  ], dtype=float32)

In [11]:
candidate_docs["cross_score"] = cross_scores

candidate_docs

,document,bi_score,cross_score
0,Artificial Intelligence is the simulation of h...,0.521461,-0.207306
4,Python is one of the most popular programming ...,0.448475,-10.088572
1,Machine Learning is a subset of Artificial Int...,0.405643,-0.014958
3,Natural Language Processing helps computers un...,0.292249,-10.786014
2,Deep Learning uses neural networks with multip...,0.233455,-9.772631


In [12]:
reranked_results = candidate_docs.sort_values(by="cross_score", ascending=False)

reranked_results

,document,bi_score,cross_score
1,Machine Learning is a subset of Artificial Int...,0.405643,-0.014958
0,Artificial Intelligence is the simulation of h...,0.521461,-0.207306
2,Deep Learning uses neural networks with multip...,0.233455,-9.772631
4,Python is one of the most popular programming ...,0.448475,-10.088572
3,Natural Language Processing helps computers un...,0.292249,-10.786014


In [13]:
def rerank_documents(query, candidate_documents, reranker_model):

    pairs = [(query, doc) for doc in candidate_documents["document"]]

    scores = reranker_model.predict(pairs)

    output = candidate_documents.copy()

    output["cross_score"] = scores

    output = output.sort_values(by="cross_score", ascending=False)

    return output

In [14]:
rerank_documents(
    query=query, candidate_documents=candidate_docs, reranker_model=cross_encoder
)

,document,bi_score,cross_score
1,Machine Learning is a subset of Artificial Int...,0.405643,-0.014958
0,Artificial Intelligence is the simulation of h...,0.521461,-0.207306
2,Deep Learning uses neural networks with multip...,0.233455,-9.772631
4,Python is one of the most popular programming ...,0.448475,-10.088572
3,Natural Language Processing helps computers un...,0.292249,-10.786014


In [15]:
def production_search(
    query,
    bi_encoder,
    cross_encoder,
    documents,
    document_embeddings,
    top_k_retrieve=5,
    top_k_final=3,
):

    # Stage 1: Retrieve
    candidates = retrieve_candidates(
        query=query,
        documents=documents,
        embeddings=document_embeddings,
        model=bi_encoder,
        top_k=top_k_retrieve,
    )

    # Stage 2: Rerank
    reranked = rerank_documents(
        query=query, candidate_documents=candidates, reranker_model=cross_encoder
    )

    return reranked.head(top_k_final)

In [16]:
production_search(
    query="How can I learn AI?",
    bi_encoder=bi_encoder,
    cross_encoder=cross_encoder,
    documents=documents,
    document_embeddings=document_embeddings,
)

,document,bi_score,cross_score
4,Python is one of the most popular programming ...,0.528897,-1.609222
1,Machine Learning is a subset of Artificial Int...,0.478066,-4.321978
0,Artificial Intelligence is the simulation of h...,0.554195,-7.559503


In [17]:
while True:

    query = input("\nEnter Query (or quit): ")

    if query.lower() == "quit":
        break

    results = production_search(
        query=query,
        bi_encoder=bi_encoder,
        cross_encoder=cross_encoder,
        documents=documents,
        document_embeddings=document_embeddings,
    )

    print("\nRe-ranked Results:\n")
    print(results)


Re-ranked Results:

                                            document  bi_score  cross_score
0  Artificial Intelligence is the simulation of h...  0.146991    -9.898018
1  Machine Learning is a subset of Artificial Int...  0.239025   -10.084352
4  Python is one of the most popular programming ...  0.110510   -10.904848
